# RouteWise separate training calibration and test experiment

The desk already works. This experiment strengthens the evidence for deciding which
tickets to route automatically. It trains a **new adapter from the original base model**.
It does not overwrite the original model or change the original assignment results.

Use a **fresh T4 Colab runtime**. Run the cells in order. The original backup ZIP is
uploaded again; Google Drive is not required. Keep the original desk notebook saved.

**Protocol fixed before this run:** seed 20261108; stratified 60/20/20 split; 3 training
epochs; LoRA rank 8; learning rate 5e-5. Only training examples update model weights.
Choose a review policy on calibration examples, aiming for at least 90% observed
automatic-route accuracy with at least 20 accepted examples. If none qualifies,
route everything to review. This is a selection rule, not an accuracy guarantee.
Evaluate the frozen policy on the test partition once, including sample counts and
an approximate 95% Wilson interval. Fit a simple TF-IDF classifier on training data.

The source dataset and its earlier results have been inspected already. This is a new
model-held-out split of the same dataset, not an independent collection of real tickets.
Do not repeatedly change settings after viewing test results. Synthetic replay remains
a separate demonstration and is not benchmark evidence.


In [ ]:
#@title 1. Install inference and notebook dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.8.0", "peft==0.18.1", "accelerate==1.11.0",
    "safetensors>=0.4.3", "scikit-learn>=1.5,<2",
    "pandas>=2.2,<3", "ipywidgets>=8.1,<9", "matplotlib>=3.9,<4",
])
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime and rerun."
x = torch.randn(32, 32, device="cuda")
_ = x @ x
torch.cuda.synchronize()
del x, _
print("GPU computation passed:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Continue to cell 2. If Colab requests a session restart, restart and rerun this cell first.")


In [ ]:
#@title 2. Upload your original backup ZIP
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile, sys
print("Select RouteWise_backup_20261107_065144_745885.zip (the duplicate (1) is also accepted).")
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, "Upload exactly one backup ZIP."
archive = archives[0]
del uploaded
expected_hash = "19b99c7092b87829509ee57f1f97669374a22128ad5c2857644187255919fdc4"
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest == expected_hash, "This differs from the verified original backup. Stop and check the file."
BACKUP_DIR = Path('/content/routewise_restored')
BACKUP_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as zipped:
    assert zipped.testzip() is None, "ZIP integrity check failed."
    for item in zipped.infolist():
        # Intermediate optimizer checkpoints are not needed for inference.
        if item.filename.startswith('adapter/checkpoint-'):
            continue
        if item.filename.endswith(('.pt', '.bin')):
            continue
        target = (BACKUP_DIR / item.filename).resolve()
        assert target.is_relative_to(BACKUP_DIR.resolve()), "Unsafe archive path."
        zipped.extract(item, BACKUP_DIR)
ADAPTER_DIR = BACKUP_DIR / 'adapter'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists()
original_metrics = json.loads((BACKUP_DIR / 'metrics.json').read_text())
print('Backup restored and fingerprint verified.')
print('Original measured accuracy:', f"{original_metrics['finetuned_accuracy']:.1%}")


In [ ]:
#@title Install the included RouteWise source files (no extra upload needed)
from pathlib import Path
import json, sys
package_files = {'routewise/__init__.py': '"""RouteWise: inspectable ticket routing and suspected-incident replay."""\n', 'routewise/core.py': '"""Pure Python routing policy, review records, and lexical incident grouping.\n\nNo model calls, network access, or simulated model scores live in this module.\n"""\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timedelta, timezone\nimport math\nimport re\nfrom typing import Mapping\n\nLABELS = (\n    "Support general", "Fileservice", "O365", "EOL", "Software",\n    "Active Directory", "Computer-Services",\n)\nSYSTEM_PROMPT = (\n    "You are an IT helpdesk ticket routing assistant. "\n    "Given a support ticket, respond with exactly one of the following categories: "\n    "Support general, Fileservice, O365, EOL, Software, Active Directory, Computer-Services."\n)\n\n\ndef normalize_scores(log_scores: Mapping[str, float], temperature: float = 1.0) -> dict:\n    if set(log_scores) != set(LABELS):\n        raise ValueError("Scores must contain each of the seven labels exactly once.")\n    if not math.isfinite(temperature) or temperature <= 0:\n        raise ValueError("Temperature must be finite and positive.")\n    values = {k: float(v) for k, v in log_scores.items()}\n    if not all(math.isfinite(v) for v in values.values()):\n        raise ValueError("Non-finite model score; send this ticket for review.")\n    top = max(values.values())\n    weights = {k: math.exp((v - top) / temperature) for k, v in values.items()}\n    total = sum(weights.values())\n    return {k: v / total for k, v in weights.items()}\n\n\ndef validate_scores(scores: Mapping[str, float]) -> None:\n    if set(scores) != set(LABELS):\n        raise ValueError("Missing or unknown routing label.")\n    if not all(math.isfinite(v) and 0 <= v <= 1 for v in scores.values()):\n        raise ValueError("Invalid routing score.")\n    if not math.isclose(sum(scores.values()), 1, abs_tol=1e-5):\n        raise ValueError("Routing scores must sum to one.")\n\n\n@dataclass(frozen=True)\nclass Policy:\n    threshold: float = 0.80\n    min_margin: float = 0.20\n    auto_enabled: bool = False\n\n    def __post_init__(self):\n        if not all(math.isfinite(v) and 0 <= v <= 1\n                   for v in (self.threshold, self.min_margin)):\n            raise ValueError("Threshold and margin must be between zero and one.")\n\n    def decide(self, scores: Mapping[str, float]) -> dict:\n        validate_scores(scores)\n        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)\n        label, score = ranked[0]\n        margin = score - ranked[1][1]\n        automatic = self.auto_enabled and score >= self.threshold and margin >= self.min_margin\n        if not self.auto_enabled:\n            reason = "Automatic routing is disabled until you enable the experimental policy."\n        elif score < self.threshold:\n            reason = "Top candidate score is below the review threshold."\n        elif margin < self.min_margin:\n            reason = "The two leading queues are too close."\n        else:\n            reason = "Meets the experimental score and margin thresholds."\n        return {\n            "predicted_queue": label, "top_score": score, "margin": margin,\n            "status": "automatic" if automatic else "needs_review", "reason": reason,\n            "policy": {"threshold": self.threshold, "min_margin": self.min_margin,\n                       "auto_enabled": self.auto_enabled, "validated": False},\n        }\n\n\ndef policy_metrics(records: list[dict], policy: Policy) -> dict:\n    if not records:\n        raise ValueError("No scored examples supplied.")\n    accepted = []\n    for row in records:\n        if row["truth"] not in LABELS:\n            raise ValueError("Evaluation truth must be a routing label.")\n        decision = policy.decide(row["scores"])\n        if decision["status"] == "automatic":\n            accepted.append(decision["predicted_queue"] == row["truth"])\n    n = len(accepted)\n    return {\n        "total": len(records), "automated": n, "review": len(records) - n,\n        "coverage": n / len(records),\n        "automatic_accuracy": sum(accepted) / n if n else None,\n        "automatic_errors": n - sum(accepted),\n    }\n\n\n@dataclass\nclass Desk:\n    tickets: list = field(default_factory=list)\n    reviews: list = field(default_factory=list)\n\n    def add(self, text: str, scores: dict, policy: Policy, timestamp=None, source="manual"):\n        if not text.strip():\n            raise ValueError("Enter a ticket first.")\n        timestamp = timestamp or datetime.now(timezone.utc)\n        if timestamp.tzinfo is None:\n            raise ValueError("Ticket timestamps need a timezone.")\n        decision = policy.decide(scores)\n        record = {\n            "id": f"T{len(self.tickets) + 1:04d}", "text": text,\n            "timestamp": timestamp.isoformat(), "source": source,\n            "scores": dict(scores), **decision,\n            "human_queue": None,\n        }\n        self.tickets.append(record)\n        return record\n\n    def review(self, ticket_id: str, queue: str):\n        if queue not in LABELS:\n            raise ValueError("Choose a valid queue.")\n        ticket = next((x for x in self.tickets if x["id"] == ticket_id), None)\n        if ticket is None:\n            raise ValueError("Ticket not found.")\n        previous = ticket["human_queue"]\n        ticket["human_queue"] = queue\n        ticket["status"] = "reviewed"\n        self.reviews.append({\n            "ticket_id": ticket_id, "queue": queue, "previous_human_queue": previous,\n            "model_queue": ticket["predicted_queue"],\n            "timestamp": datetime.now(timezone.utc).isoformat(),\n        })\n        return ticket\n\n\n_STOPWORDS = set("i my me we our the a an to of and is are it in on for with since please can cannot not has have this all".split())\n_ALIASES = {\n    "outlook": "email", "mailbox": "email", "mailboxes": "email", "emails": "email",\n    "mail": "email", "messages": "email", "message": "email",\n    "sending": "send", "sent": "send", "sends": "send",\n    "failing": "failure", "failed": "failure", "fails": "failure",\n    "stopped": "failure", "broken": "failure", "down": "failure",\n}\n\n\ndef ticket_terms(text: str) -> Counter:\n    words = re.findall(r"[a-z0-9]+", text.lower())\n    return Counter(_ALIASES.get(w, w) for w in words if w not in _STOPWORDS)\n\n\ndef similarity(a: Counter, b: Counter) -> float:\n    denominator = math.sqrt(sum(v*v for v in a.values()) * sum(v*v for v in b.values()))\n    return sum(v * b.get(k, 0) for k, v in a.items()) / denominator if denominator else 0.0\n\n\ndef suspected_incidents(tickets: list[dict], now=None, window_minutes=10,\n                        min_tickets=3, similarity_threshold=0.55) -> list[dict]:\n    """Complete-link grouping avoids merging dissimilar endpoints through a bridge.\n\n    Lexical similarity with documented email aliases; not root-cause diagnosis.\n    Each distinct ticket counts once. Synthetic reporters do not prove real impact.\n    """\n    if window_minutes <= 0 or min_tickets < 2 or not 0 < similarity_threshold <= 1:\n        raise ValueError("Invalid incident grouping settings.")\n    if not tickets:\n        return []\n    times = [datetime.fromisoformat(x["timestamp"]) for x in tickets]\n    if any(x.tzinfo is None for x in times):\n        raise ValueError("Incident timestamps need a timezone.")\n    now = now or max(times)\n    if now.tzinfo is None:\n        raise ValueError("Current time needs a timezone.")\n    recent = {}\n    for ticket, time in zip(tickets, times):\n        if now - timedelta(minutes=window_minutes) <= time <= now:\n            recent[ticket["id"]] = ticket\n    groups = []\n    for ticket in sorted(recent.values(), key=lambda x: (x["timestamp"], x["id"])):\n        terms = ticket_terms(ticket["text"])\n        match = next((g for g in groups if all(\n            similarity(terms, ticket_terms(other["text"])) >= similarity_threshold\n            for other in g)), None)\n        if match is None:\n            groups.append([ticket])\n        else:\n            match.append(ticket)\n    return [{\n        "title": "Suspected shared incident", "ticket_ids": [x["id"] for x in group],\n        "count": len(group), "example": group[0]["text"],\n        "method": "Lexical similarity with email aliases; complete-link grouping",\n        "confirmed": False,\n    } for group in groups if len(group) >= min_tickets]\n', 'routewise/demo.py': '"""A notebook-native desk. No public server or external ticket integrations."""\nfrom datetime import datetime, timedelta, timezone\nfrom html import escape\nfrom pathlib import Path\nimport json\nimport ipywidgets as w\nfrom IPython.display import display, HTML, clear_output\nfrom .core import Desk, Policy, LABELS, suspected_incidents\n\nSCENARIOS = [\n    ("Please install the approved Adobe Acrobat application on my laptop.", "Software"),\n    ("I cannot access the shared folder through Teams. Please investigate.", None),\n    ("Outlook email sending failed for our team this morning.", "O365"),\n    ("Our team email sending failed in Outlook this morning.", "O365"),\n    ("Outlook email sending has failed again for our team this morning.", "O365"),\n    ("Please decommission the retired test server and remove it from monitoring.", "EOL"),\n    ("Please create an Active Directory account for the new employee.", "Active Directory"),\n    ("The office printer has a persistent paper jam and needs repair.", "Computer-Services"),\n]\n\n\ndef launch_desk(router):\n    desk = Desk()\n    output = w.Output()\n    text = w.Textarea(placeholder="Describe the support request…", layout=w.Layout(width="100%", height="90px"))\n    threshold = w.FloatSlider(value=.80, min=.15, max=.99, step=.01,\n                              description="Score cutoff", style={"description_width": "initial"})\n    margin = w.FloatSlider(value=.20, min=0, max=.8, step=.01,\n                           description="Top-two gap", style={"description_width": "initial"})\n    automatic = w.Checkbox(value=False, description="Enable experimental automatic routing")\n    route = w.Button(description="Route ticket", button_style="primary")\n    replay = w.Button(description="Replay 8 synthetic tickets")\n    refresh = w.Button(description="Refresh desk")\n    export = w.Button(description="Download desk records")\n    ticket_select = w.Dropdown(options=[], description="Ticket")\n    queue_select = w.Dropdown(options=LABELS, description="Human queue")\n    review = w.Button(description="Save human decision")\n    notice = w.HTML()\n\n    def policy():\n        return Policy(threshold.value, margin.value, automatic.value)\n\n    def table(headers, rows):\n        return "<table style=\'width:100%;border-collapse:collapse\'>" + "<tr>" + "".join(\n            f"<th style=\'text-align:left;padding:8px;border-bottom:1px solid #aaa\'>{escape(str(h))}</th>"\n            for h in headers) + "</tr>" + "".join("<tr>" + "".join(\n                f"<td style=\'padding:8px;border-bottom:1px solid #ddd\'>{escape(str(v))}</td>"\n                for v in row) + "</tr>" for row in rows) + "</table>"\n\n    def render():\n        ticket_select.options = [(f"{r[\'id\']} · {r[\'status\']}", r[\'id\']) for r in desk.tickets]\n        with output:\n            clear_output(wait=True)\n            auto = sum(r["status"] == "automatic" for r in desk.tickets)\n            pending = sum(r["status"] == "needs_review" for r in desk.tickets)\n            display(HTML(f"<h3>Ticket desk</h3><p>{len(desk.tickets)} tickets · {auto} automatic · "\n                         f"{pending} awaiting review · {len(desk.reviews)} human decisions</p>"))\n            display(HTML("<p>Scores are relative preferences across seven labels, not validated probabilities "\n                         "of correctness. Threshold changes affect new decisions only.</p>"))\n            if desk.tickets:\n                display(HTML(table(["ID", "Ticket", "Suggested queue", "Status", "Human queue", "Source"], [\n                    [r[\'id\'], r[\'text\'], r[\'predicted_queue\'] or \'Inference failed\', r[\'status\'],\n                     r[\'human_queue\'] or \'—\', r[\'source\']] for r in desk.tickets])))\n                latest = desk.tickets[-1]\n                if latest[\'scores\']:\n                    display(HTML("<h3>Latest ticket candidate scores</h3>" + table(\n                        ["Queue", "Relative score"],\n                        [(k, f"{v:.3f}") for k, v in sorted(latest[\'scores\'].items(), key=lambda x:x[1], reverse=True)])))\n                display(HTML(f"<p>{escape(latest[\'reason\'])}</p>"))\n                incidents = suspected_incidents(desk.tickets)\n                display(HTML("<h3>Incident Radar</h3><p>Lexical grouping over a 10-minute window; "\n                             "three reports required. Groups are suspected, not confirmed outages.</p>"))\n                for group in incidents:\n                    display(HTML(f"<p><b>{escape(group[\'title\'])}</b>: {group[\'count\']} reports · "\n                                 f"{escape(\', \'.join(group[\'ticket_ids\']))}<br>{escape(group[\'example\'])}</p>"))\n                if not incidents:\n                    display(HTML("<p>No group currently meets the incident settings.</p>"))\n\n    def add(ticket, timestamp=None, source="manual"):\n        try:\n            result = router.score(ticket)\n            record = desk.add(ticket, result[\'scores\'], policy(), timestamp, source)\n            record.update({k: result[k] for k in [\'log_scores\', \'latency_ms\', \'score_method\', \'calibrated\']})\n        except Exception as error:\n            # Preserve inference failures as review items, with no invented label or score.\n            record = {\n                \'id\': f\'T{len(desk.tickets)+1:04d}\', \'text\': ticket,\n                \'timestamp\': (timestamp or datetime.now(timezone.utc)).isoformat(),\n                \'source\': source, \'scores\': {}, \'predicted_queue\': None,\n                \'human_queue\': None, \'status\': \'needs_review\',\n                \'reason\': f\'Inference failed: {type(error).__name__}: {error}\',\n            }\n            desk.tickets.append(record)\n        return record\n\n    def on_route(_):\n        if not text.value.strip():\n            notice.value = "Enter a ticket first."\n            return\n        route.disabled = replay.disabled = True\n        notice.value = "Scoring all seven complete labels…"\n        try:\n            add(text.value)\n            render()\n            notice.value = "Ticket processed."\n        finally:\n            route.disabled = replay.disabled = False\n\n    def on_replay(_):\n        route.disabled = replay.disabled = True\n        start = datetime.now(timezone.utc) - timedelta(minutes=4)\n        try:\n            for i, (ticket, expected) in enumerate(SCENARIOS):\n                notice.value = f"Scoring synthetic ticket {i+1}/{len(SCENARIOS)}…"\n                record = add(ticket, start + timedelta(seconds=30*i), \'synthetic_replay\')\n                record[\'illustrative_expected_queue\'] = expected\n                render()\n            notice.value = "Synthetic replay complete. These examples are not a benchmark."\n        finally:\n            route.disabled = replay.disabled = False\n\n    def on_review(_):\n        if ticket_select.value:\n            desk.review(ticket_select.value, queue_select.value)\n            render()\n            notice.value = "Human decision saved. Model weights have not changed."\n\n    def on_export(_):\n        path = Path(\'/content/routewise_desk_records.json\')\n        path.write_text(json.dumps({\n            \'tickets\': desk.tickets, \'reviews\': desk.reviews,\n            \'notice\': \'Experimental policy; synthetic replay is not benchmark evidence.\',\n        }, indent=2), encoding=\'utf-8\')\n        from google.colab import files\n        files.download(str(path))\n\n    route.on_click(on_route)\n    replay.on_click(on_replay)\n    refresh.on_click(lambda _: render())\n    review.on_click(on_review)\n    export.on_click(on_export)\n    display(w.VBox([\n        w.HTML("<h2>RouteWise Incident Desk</h2><p>Your restored Qwen LoRA model · "\n               "experimental review policy · notebook session storage</p>"),\n        text, w.HBox([threshold, margin]), automatic,\n        w.HBox([route, replay, refresh, export]), notice,\n        w.HBox([ticket_select, queue_select, review]), output,\n    ]))\n    render()\n    return desk\n', 'routewise/experiment.py': '"""Reproducible data preparation and calibration-only policy selection."""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nfrom .core import LABELS, Policy, policy_metrics\n\n\ndef text_id(text):\n    normalized = re.sub(r\'\\s+\', \' \', text.strip().casefold())\n    return hashlib.sha256(normalized.encode(\'utf-8\')).hexdigest()\n\n\ndef recover_dataset(backup_dir):\n    """Reconstruct the dataset from the original 468 train + 117 validation rows."""\n    root = Path(backup_dir)\n    records = json.loads((root/\'TRAIN.json\').read_text())\n    rows = []\n    for record in records:\n        users = [m[\'content\'] for m in record[\'messages\'] if m[\'role\'] == \'user\']\n        labels = [m[\'content\'] for m in record[\'messages\'] if m[\'role\'] == \'assistant\']\n        if len(users) != 1 or len(labels) != 1 or not users[0].startswith(\'Support Ticket: \'):\n            raise ValueError(\'Unexpected original training-record format.\')\n        rows.append({\'text\':users[0].removeprefix(\'Support Ticket: \'), \'label\':labels[0]})\n    with (root/\'val_split.csv\').open(newline=\'\') as file:\n        rows.extend({\'text\':r[\'text\'], \'label\':r[\'label\']} for r in csv.DictReader(file))\n    seen = set()\n    for row in rows:\n        if row[\'label\'] not in LABELS or not row[\'text\'].strip():\n            raise ValueError(\'Missing text or invalid routing label.\')\n        row[\'id\'] = text_id(row[\'text\'])\n        if row[\'id\'] in seen:\n            raise ValueError(\'Duplicate normalized ticket text: group duplicates before splitting.\')\n        seen.add(row[\'id\'])\n    return sorted(rows, key=lambda row: row[\'id\'])\n\n\ndef split_dataset(rows, seed=20261108):\n    from sklearn.model_selection import train_test_split\n    training, remainder = train_test_split(\n        rows, test_size=.4, stratify=[r[\'label\'] for r in rows], random_state=seed)\n    calibration, test = train_test_split(\n        remainder, test_size=.5, stratify=[r[\'label\'] for r in remainder], random_state=seed)\n    partitions = {\'train\':training, \'calibration\':calibration, \'test\':test}\n    ids = {name: {r[\'id\'] for r in split} for name, split in partitions.items()}\n    assert not (ids[\'train\'] & ids[\'calibration\'] or ids[\'train\'] & ids[\'test\']\n                or ids[\'calibration\'] & ids[\'test\'])\n    for split in partitions.values():\n        assert {r[\'label\'] for r in split} == set(LABELS)\n    assert sum(map(len, partitions.values())) == len(rows)\n    return partitions\n\n\ndef wilson_interval(correct, total, z=1.959963984540054):\n    if total < 0 or correct < 0 or correct > total:\n        raise ValueError(\'Invalid success counts.\')\n    if total == 0:\n        return None\n    p = correct/total\n    denominator = 1+z*z/total\n    center = (p+z*z/(2*total))/denominator\n    radius = z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/denominator\n    return [max(0, center-radius), min(1, center+radius)]\n\n\ndef choose_policy(calibration_rows, target_accuracy=.90, min_automated=20):\n    """A fixed finite grid. Uses calibration rows only, with a review-all fallback.\n\n    The target is an empirical selection rule, NOT an accuracy guarantee.\n    """\n    if not .5 <= target_accuracy <= 1 or min_automated < 1:\n        raise ValueError(\'Invalid selection target.\')\n    if not calibration_rows:\n        raise ValueError(\'Calibration examples are required.\')\n    candidates = []\n    for threshold in [.40, .50, .60, .70, .80, .85, .90, .95, .99]:\n        for margin in [0, .10, .20, .30]:\n            policy = Policy(threshold, margin, True)\n            metrics = policy_metrics(calibration_rows, policy)\n            candidates.append({\'threshold\':threshold, \'min_margin\':margin,\n                               \'auto_enabled\':True, **metrics})\n    eligible = [c for c in candidates if c[\'automated\'] >= min_automated\n                and c[\'automatic_accuracy\'] >= target_accuracy]\n    if eligible:\n        best = max(eligible, key=lambda c:(c[\'automated\'], -c[\'automatic_errors\'],\n                                          c[\'threshold\'], c[\'min_margin\']))\n        chosen = {k:best[k] for k in [\'threshold\',\'min_margin\',\'auto_enabled\']}\n        calibration_metrics = policy_metrics(calibration_rows, Policy(**chosen))\n    else:\n        chosen = {\'threshold\':1.0, \'min_margin\':1.0, \'auto_enabled\':False}\n        calibration_metrics = policy_metrics(calibration_rows, Policy(**chosen))\n    return {\n        \'policy\':chosen, \'selection_split\':\'calibration\', \'target_accuracy\':target_accuracy,\n        \'min_automated\':min_automated, \'target_is_guarantee\':False,\n        \'fallback_review_all\':not bool(eligible),\n        \'calibration_metrics\':calibration_metrics, \'grid\':candidates,\n    }\n', 'routewise/inference.py': '"""GPU inference, imported only after the Colab dependency cell has run."""\nfrom pathlib import Path\nfrom time import perf_counter\nimport math\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom peft import PeftModel\nfrom .core import LABELS, SYSTEM_PROMPT, normalize_scores\n\nBASE_MODEL = "Qwen/Qwen3-1.7B-Base"\n# Recorded in the successful original training log; do not silently change weights.\nBASE_REVISION = "ea980cb0a6c2ae4b936e82123acc929f1cec04c1"\n\n\nclass Router:\n    def __init__(self, model, tokenizer, max_tokens=2048):\n        self.model, self.tokenizer = model.eval(), tokenizer\n        self.max_tokens = max_tokens\n        if not tokenizer.chat_template:\n            raise ValueError("The saved adapter tokenizer must contain its chat template.")\n        if tokenizer.eos_token_id is None:\n            raise ValueError("Saved tokenizer has no EOS token.")\n        self.candidates = {\n            label: tokenizer.encode(label, add_special_tokens=False) + [tokenizer.eos_token_id]\n            for label in LABELS\n        }\n\n    @classmethod\n    def restore(cls, adapter_dir):\n        adapter_dir = Path(adapter_dir)\n        if not torch.cuda.is_available():\n            raise RuntimeError("Choose a T4 GPU runtime before loading the model.")\n        tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir), local_files_only=True)\n        base = AutoModelForCausalLM.from_pretrained(\n            BASE_MODEL, revision=BASE_REVISION,\n            torch_dtype=torch.float16, device_map={"": 0},\n            attn_implementation="sdpa",\n        )\n        model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)\n        # Training replaced EOS with im_end. Keep restored generation consistent.\n        model.config.eos_token_id = tokenizer.eos_token_id\n        model.config.pad_token_id = tokenizer.pad_token_id\n        model.generation_config.eos_token_id = tokenizer.eos_token_id\n        model.generation_config.pad_token_id = tokenizer.pad_token_id\n        return cls(model, tokenizer)\n\n    def _prompt_ids(self, text):\n        if not isinstance(text, str) or not text.strip():\n            raise ValueError("Ticket must contain text.")\n        if "<|im_start|>" in text or "<|im_end|>" in text:\n            raise ValueError("Ticket contains chat control tokens; send for manual review.")\n        messages = [\n            {"role": "system", "content": SYSTEM_PROMPT},\n            {"role": "user", "content": f"Support Ticket: {text}"},\n        ]\n        # Do not pass enable_thinking=False: that saved Qwen Jinja template adds\n        # an empty think block, whereas qwen3_nothink training used a plain prefix.\n        prompt = self.tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True,\n        )\n        if not prompt.endswith("<|im_start|>assistant\\n"):\n            raise ValueError("Unexpected generation prefix; check the saved tokenizer template.")\n        ids = self.tokenizer.encode(prompt, add_special_tokens=False)\n        if len(ids) + max(map(len, self.candidates.values())) > self.max_tokens:\n            raise ValueError("Ticket exceeds the training token limit; manual review required.")\n        return ids\n\n    @torch.inference_mode()\n    def score(self, text, temperature=1.0):\n        prompt = self._prompt_ids(text)\n        torch.cuda.synchronize()\n        start = perf_counter()\n        raw = {}\n        for label, completion in self.candidates.items():\n            ids = torch.tensor([prompt + completion], device=self.model.device)\n            mask = torch.ones_like(ids)\n            # Retain only positions that predict the complete label and EOS.\n            # Sequential candidates bound peak GPU memory for the large vocabulary.\n            logits = self.model(\n                input_ids=ids, attention_mask=mask, use_cache=False,\n                logits_to_keep=len(completion) + 1,\n            ).logits[0, :-1, :].float()\n            targets = torch.tensor(completion, device=logits.device)\n            token_scores = logits.log_softmax(-1).gather(-1, targets[:, None]).squeeze(-1)\n            raw[label] = token_scores.sum().item()\n            del logits, token_scores\n        torch.cuda.synchronize()\n        scores = normalize_scores(raw, temperature=temperature)\n        return {\n            "prediction": max(scores, key=scores.get), "scores": scores,\n            "log_scores": raw, "latency_ms": (perf_counter() - start) * 1000,\n            "score_method": "Sum of complete-label and EOS log likelihoods",\n            "calibrated": False, "temperature": temperature,\n        }\n\n    @torch.inference_mode()\n    def generate_label(self, text):\n        prompt = self._prompt_ids(text)\n        ids = torch.tensor([prompt], device=self.model.device)\n        output = self.model.generate(\n            input_ids=ids, attention_mask=torch.ones_like(ids),\n            max_new_tokens=24, do_sample=False, use_cache=True,\n            eos_token_id=self.tokenizer.eos_token_id,\n            pad_token_id=self.tokenizer.pad_token_id,\n        )\n        generated = self.tokenizer.decode(output[0, len(prompt):], skip_special_tokens=True).strip()\n        # Never silently convert a malformed generation into Support general.\n        return {"raw_output": generated, "valid": generated in LABELS,\n                "prediction": generated if generated in LABELS else None}\n'}
for relative, source in package_files.items():
    target = Path("/content") / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
if "/content" not in sys.path: sys.path.insert(0, "/content")
print("RouteWise modules ready.")


In [ ]:
#@title A. Prepare and save the three separate partitions
from routewise.experiment import recover_dataset, split_dataset
from routewise.core import LABELS, Policy, policy_metrics
from routewise.inference import Router, BASE_MODEL, BASE_REVISION
from pathlib import Path
import pandas as pd
import json, hashlib, gc
SEED = 20261108
RUN_DIR = Path('/content/routewise_separate_evaluation')
RUN_DIR.mkdir(exist_ok=True)
partitions = split_dataset(recover_dataset(BACKUP_DIR), seed=SEED)
assert [len(partitions[k]) for k in ['train','calibration','test']] == [351,117,117]
for name, rows in partitions.items():
    pd.DataFrame(rows).to_csv(RUN_DIR / f'{name}_split.csv', index=False)
manifest = {
    'seed':SEED, 'base_model':BASE_MODEL, 'base_revision':BASE_REVISION,
    'original_adapter_used_for_training':False,
    'prior_dataset_exposure':'Original results and examples already inspected; not external validation.',
    'split_ids':{k:[r['id'] for r in v] for k,v in partitions.items()},
    'counts':{k:len(v) for k,v in partitions.items()},
    'protocol':{'epochs':3, 'learning_rate':5e-5, 'lora_rank':8,
                'calibration_target':.90, 'calibration_min_automated':20},
}
(RUN_DIR/'split_manifest.json').write_text(json.dumps(manifest, indent=2))
display(pd.DataFrame({name:pd.Series([r['label'] for r in rows]).value_counts()
                      for name,rows in partitions.items()}).fillna(0).astype(int))
print('Disjoint by normalized text ID. Counts: 351 train / 117 calibration / 117 test.')
print('Near-duplicate meaning is not automatically detected by this check.')


In [ ]:
#@title B. Fit the simple text-classification comparator (training examples only)
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
classical = make_pipeline(
    TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
    LogisticRegression(C=1.0, max_iter=2000, random_state=SEED),
)
classical.fit([r['text'] for r in partitions['train']],
              [r['label'] for r in partitions['train']])
print('TF-IDF + logistic regression fitted. Test predictions will be measured after policy freeze.')


In [ ]:
#@title C. Train a separate adapter from the base model (one new training run)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model
set_seed(SEED)
NEW_ADAPTER = RUN_DIR/'adapter'
assert not (NEW_ADAPTER/'adapter_config.json').exists(), (
    'A trained extension adapter is already saved here. Do not retrain to chase test results. '
    'If this cell completed previously, continue to calibration.')
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), local_files_only=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, torch_dtype=torch.float16,
    attn_implementation='sdpa',
)
base.config.use_cache = False
base.config.eos_token_id = tokenizer.eos_token_id
base.config.pad_token_id = tokenizer.pad_token_id
train_model = get_peft_model(base, LoraConfig(
    task_type='CAUSAL_LM', r=8, lora_alpha=16, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
))
formatter = Router(train_model, tokenizer)
examples = []
for row in partitions['train']:
    prompt_ids = formatter._prompt_ids(row['text'])
    answer_ids = formatter.candidates[row['label']]
    examples.append({
        'input_ids':prompt_ids+answer_ids,
        'labels':[-100]*len(prompt_ids)+answer_ids,
    })
def collate(batch):
    length = max(len(row['input_ids']) for row in batch)
    return {
        'input_ids':torch.tensor([row['input_ids']+[tokenizer.pad_token_id]*(length-len(row['input_ids'])) for row in batch]),
        'attention_mask':torch.tensor([[1]*len(row['input_ids'])+[0]*(length-len(row['input_ids'])) for row in batch]),
        'labels':torch.tensor([row['labels']+[-100]*(length-len(row['labels'])) for row in batch]),
    }
arguments = TrainingArguments(
    output_dir=str(RUN_DIR/'training'), num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=5e-5, lr_scheduler_type='cosine', warmup_steps=0,
    fp16=True, bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},
    logging_steps=5, save_strategy='no', report_to='none',
    seed=SEED, data_seed=SEED, dataloader_num_workers=0,
)
trainer = Trainer(model=train_model, args=arguments, train_dataset=examples,
                  data_collator=collate, processing_class=tokenizer)
train_model.train()
trained = trainer.train()
trainer.save_model(str(NEW_ADAPTER))
tokenizer.save_pretrained(str(NEW_ADAPTER))
trainer.save_state()
(RUN_DIR/'training_metrics.json').write_text(json.dumps(trained.metrics, indent=2))
(RUN_DIR/'training_log.json').write_text(json.dumps(trainer.state.log_history, indent=2))
train_model.gradient_checkpointing_disable()
train_model.config.use_cache = True
train_model.generation_config.eos_token_id = tokenizer.eos_token_id
train_model.generation_config.pad_token_id = tokenizer.pad_token_id
evaluation_router = Router(train_model, tokenizer)
print('New adapter trained and saved separately. Original adapter has not been modified.')


In [ ]:
#@title D. Choose and freeze the review policy using calibration examples only
from routewise.experiment import choose_policy
from tqdm.auto import tqdm
from datetime import datetime, timezone
assert not (RUN_DIR/'test_results.json').exists(), 'Test results already exist. Do not retune after viewing them.'
calibration_rows = []
for row in tqdm(partitions['calibration'], desc='Calibration scoring'):
    result = evaluation_router.score(row['text'])
    calibration_rows.append({'id':row['id'], 'truth':row['label'], **result})
choice = choose_policy(calibration_rows, target_accuracy=.90, min_automated=20)
choice['frozen_at_utc'] = datetime.now(timezone.utc).isoformat()
weights = NEW_ADAPTER/'adapter_model.safetensors'
choice['adapter_sha256'] = hashlib.sha256(weights.read_bytes()).hexdigest()
choice['split_manifest_sha256'] = hashlib.sha256((RUN_DIR/'split_manifest.json').read_bytes()).hexdigest()
frozen_text = json.dumps(choice, indent=2)
(RUN_DIR/'frozen_policy.json').write_text(frozen_text)
frozen_policy_hash = hashlib.sha256(frozen_text.encode()).hexdigest()
(RUN_DIR/'frozen_policy.sha256').write_text(frozen_policy_hash)
(RUN_DIR/'calibration_scores.json').write_text(json.dumps(calibration_rows, indent=2))
selected_policy = Policy(**choice['policy'])
print('FROZEN POLICY:', choice['policy'])
print('Calibration performance:', choice['calibration_metrics'])
if choice['fallback_review_all']:
    print('No candidate met the predeclared calibration criteria: all tickets will go to review.')
print('Policy selection is complete. Test examples have not been scored in this run.')


In [ ]:
#@title E. Evaluate the frozen policy and comparators on test examples once
from sklearn.metrics import accuracy_score, classification_report
from routewise.experiment import wilson_interval
assert hashlib.sha256((RUN_DIR/'frozen_policy.json').read_bytes()).hexdigest() == frozen_policy_hash
assert hashlib.sha256((NEW_ADAPTER/'adapter_model.safetensors').read_bytes()).hexdigest() == choice['adapter_sha256']
test_rows = []
for row in tqdm(partitions['test'], desc='Final test scoring'):
    tuned = evaluation_router.score(row['text'])
    with evaluation_router.model.disable_adapter():
        baseline = evaluation_router.score(row['text'])
    test_rows.append({
        'id':row['id'], 'truth':row['label'], 'text':row['text'],
        'prediction':tuned['prediction'], 'scores':tuned['scores'],
        'log_scores':tuned['log_scores'], 'latency_ms':tuned['latency_ms'],
        'base_prediction':baseline['prediction'], 'base_latency_ms':baseline['latency_ms'],
    })
truth = [r['truth'] for r in test_rows]
simple_predictions = classical.predict([r['text'] for r in partitions['test']]).tolist()
for row, prediction in zip(test_rows, simple_predictions): row['tfidf_prediction'] = prediction
selective = policy_metrics(test_rows, selected_policy)
interval = wilson_interval(selective['automated']-selective['automatic_errors'], selective['automated'])
results = {
    'status':'New model-held-out split; source dataset previously inspected, not external validation.',
    'test_count':len(test_rows), 'policy':choice['policy'],
    'policy_sha256':frozen_policy_hash, 'policy_selected_on':'calibration',
    'base_accuracy':accuracy_score(truth, [r['base_prediction'] for r in test_rows]),
    'tuned_accuracy':accuracy_score(truth, [r['prediction'] for r in test_rows]),
    'tfidf_accuracy':accuracy_score(truth, simple_predictions),
    'selective_routing':selective, 'automatic_accuracy_wilson95':interval,
    'tuned_report':classification_report(truth, [r['prediction'] for r in test_rows],
        labels=list(LABELS), target_names=list(LABELS), output_dict=True, zero_division=0),
}
(RUN_DIR/'test_results.json').write_text(json.dumps(results, indent=2))
(RUN_DIR/'test_predictions.json').write_text(json.dumps(test_rows, indent=2))
display(pd.DataFrame([
    {'Model':'Base Qwen, full-label scoring', 'Test accuracy':results['base_accuracy']},
    {'Model':'New tuned Qwen, full-label scoring', 'Test accuracy':results['tuned_accuracy']},
    {'Model':'TF-IDF + logistic regression', 'Test accuracy':results['tfidf_accuracy']},
]))
print('Frozen-policy test results:', selective)
print('Approximate 95% interval for automatic-route accuracy:', interval)
print('Report every comparator, even if the simpler model wins. Do not retune using test results.')


In [ ]:
#@title F. Download the new adapter and experiment evidence
import zipfile
from google.colab import files
archive = Path('/content/RouteWise_separate_evaluation.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file(): zipped.write(path, path.relative_to(RUN_DIR))
with zipfile.ZipFile(archive) as zipped:
    assert zipped.testzip() is None
    assert 'test_results.json' in zipped.namelist()
    assert 'adapter/adapter_model.safetensors' in zipped.namelist()
print('Archive verified. Download it, then share it in the RouteWise chat.')
files.download(str(archive))
